In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt
from datetime import datetime, timedelta
from tqdm import tqdm

In [2]:
print(f'Last run date: {dt.datetime.today()}')

Last run date: 2024-03-15 14:57:33.480911


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 09_early_indicators
Subtask: 10_loss_forecasting


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Get a list of month end dates

In [6]:
start_date = datetime(2015, 9, 30)
end_date = datetime(2024, 2, 29)

current_date = start_date
list_str_month_end = []
while current_date <= end_date:
    next_month = current_date.replace(day=28) + timedelta(days=4)  # Move to the last day of the current month
    last_day_of_month = next_month - timedelta(days=next_month.day)
    list_str_month_end.append(last_day_of_month.strftime('%Y-%m-%d'))
    current_date = next_month
#list_str_month_end

### Pull data

In [7]:
# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

In [8]:
list_df = []
for str_month_end in tqdm(list_str_month_end):
    # read query
    str_filepath = './sql/query.sql'
    str_query = open(str_filepath, 'r').read()
    str_query = str_query.replace('STR_MONTH_END', str_month_end)
    #print(str_query)
    # read sql
    df = pd.read_sql_query(
        str_query, 
        con=conn,
    )
    df['str_month_end'] = str_month_end
    list_df.append(df)
df = pd.concat(list_df)
del list_df
conn.close()
df

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 102/102 [01:07<00:00,  1.50it/s]


,bigAccountId,dtmBooking,BookingQuarter,fltNetChgOff,MonthEndDate,str_month_end
0,239023,2004-09-16,2004 Q3,176.65,2019-08-31,2019-08-31
1,120431,2002-12-10,2002 Q4,1992.16,2019-08-31,2019-08-31
2,232646,2004-07-26,2004 Q3,11204.56,2019-08-31,2019-08-31
3,218982,2004-03-31,2004 Q1,12933.67,2019-08-31,2019-08-31
4,229275,2004-06-21,2004 Q2,17411.42,2019-08-31,2019-08-31
...,...,...,...,...,...,...
111269,163875,2003-09-09,2003 Q3,4040.52,2024-02-29,2024-02-29
111270,163924,2001-02-12,2001 Q1,1040.77,2024-02-29,2024-02-29
111271,163986,2003-09-30,2003 Q3,6853.32,2024-02-29,2024-02-29
111272,164004,2003-09-11,2003 Q3,6071.15,2024-02-29,2024-02-29


### Save

In [9]:
%%time

# save
str_filename = 'df_loss.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 6.96 s


### Upload to s3

In [10]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/00_get_loss/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 1.65 s


### Clean-up

In [11]:
os.remove(str_local_path)